# 01 — Annual FTR Data Profile

**Goal:** Profile annual cleared trades (PY 2019-2025), f0p cleared trades, and compare volumes.

Resolves assumptions: A2 (volume comparison), A3 (f0p residual magnitudes)

In [ ]:
import os
os.chdir('/home/xyz/workspace/pmodel/src')

from pbase.config.ray import init_ray
import pmodel
init_ray(extra_modules=[pmodel])

from pbase.analysis.utils import misc
misc.logging_setup()

import pandas as pd
import numpy as np

pd.options.display.float_format = '{:.2f}'.format
pd.set_option('display.max_columns', 99)
pd.set_option('display.max_rows', 200)

from pbase.analysis.tools.all_positions import MisoApTools
aptools = MisoApTools()
print('MisoApTools initialized')

## 1.1 Load Annual Cleared Trades (PY 2019-2025)

In [ ]:
# Load annual + f0p cleared trades for each PY (full year: Jun Y to May Y+1)
annual_frames = []
f0p_frames = []
for py in range(2019, 2026):
    print(f'Loading PY {py} (Jun {py} to May {py+1})...')
    start = pd.Timestamp(f'{py}-06-01')
    end = pd.Timestamp(f'{py+1}-06-01')  # exclusive end
    trades = aptools.get_all_cleared_trades(
        start_date=start,
        end_date=end
    )
    # Annual
    annual = trades[trades['period_type'].isin(['aq1','aq2','aq3','aq4'])].copy()
    annual['planning_year'] = py
    annual_frames.append(annual)
    print(f'  Annual: {len(annual)} rows')
    
    # f0p
    f0p = trades[trades['period_type'].isin(['f0','f1','q4'])].copy()
    f0p['planning_year'] = py
    f0p_frames.append(f0p)
    print(f'  f0p: {len(f0p)} rows')

annual_all = pd.concat(annual_frames, ignore_index=True)
f0p_all = pd.concat(f0p_frames, ignore_index=True)
print(f'\nTotal annual rows: {len(annual_all)}')
print(f'Total f0p rows: {len(f0p_all)}')
print(f'Annual period_types: {annual_all["period_type"].value_counts().to_dict()}')

In [ ]:
# Profile: rows by PY x round x quarter x class_type
profile = annual_all.groupby(['planning_year', 'round', 'period_type', 'class_type']).agg(
    row_count=('path', 'size'),
    unique_paths=('path', 'nunique'),
    total_mw=('cleared_volume', 'sum'),
    total_mwh=('cleared_mwh', 'sum'),
).reset_index()

print('=== Annual Cleared Trades Profile ===')
print(profile.to_string(index=False))

In [ ]:
# 3x4 grid: row counts by round x quarter (summed over class_type and PY)
grid_counts = annual_all.groupby(['round', 'period_type']).agg(
    total_rows=('path', 'size'),
    unique_paths=('path', 'nunique'),
    total_mw=('cleared_volume', 'sum'),
    avg_mw_per_row=('cleared_volume', 'mean'),
).reset_index()

print('=== 3x4 Grid: Round x Quarter (all PYs, both class_types) ===')
print(grid_counts.to_string(index=False))
print()

# Pivot for cleaner view
for metric in ['total_rows', 'total_mw']:
    pivot = grid_counts.pivot(index='round', columns='period_type', values=metric)
    pivot = pivot[['aq1','aq2','aq3','aq4']]
    print(f'\n--- {metric} by Round x Quarter ---')
    print(pivot.to_string())

In [ ]:
# Per-PY summary
py_summary = annual_all.groupby(['planning_year']).agg(
    total_rows=('path', 'size'),
    unique_paths=('path', 'nunique'),
    total_mw=('cleared_volume', 'sum'),
    total_mwh=('cleared_mwh', 'sum'),
    n_rounds=('round', 'nunique'),
).reset_index()
print('=== Per-PY Summary ===')
print(py_summary.to_string(index=False))

## 1.2 Load f0p Cleared Trades (same PYs)

In [ ]:
# Load f0p cleared trades for the same PY range
# f0 trades are monthly, so we load all months for each PY (Jun Y to May Y+1)
f0p_frames = []
for py in range(2019, 2026):
    print(f'Loading f0p for PY {py}...')
    start = pd.Timestamp(f'{py}-06-01')
    end = pd.Timestamp(f'{py+1}-05-01')
    trades = aptools.get_all_cleared_trades(
        start_date=start,
        end_date=end
    )
    f0p = trades[trades['period_type'].isin(['f0','f1','q4'])].copy()
    f0p['planning_year'] = py
    f0p_frames.append(f0p)
    print(f'  PY {py}: {len(f0p)} rows')

f0p_all = pd.concat(f0p_frames, ignore_index=True)
print(f'\nTotal f0p rows: {len(f0p_all)}')

In [ ]:
# f0p profile by PY x period_type x class_type
f0p_profile = f0p_all.groupby(['planning_year', 'period_type', 'class_type']).agg(
    row_count=('path', 'size'),
    unique_paths=('path', 'nunique'),
    total_mw=('cleared_volume', 'sum'),
    avg_mw_per_row=('cleared_volume', 'mean'),
).reset_index()

print('=== f0p Cleared Trades Profile ===')
print(f0p_profile.to_string(index=False))

## 1.3 Annual vs f0p Volume Comparison

In [ ]:
# Compare annual vs f0p: average MW per path per period, total MW per PY
# Annual: one clearing per round per quarter
# f0p: monthly clearings

annual_vol = annual_all.groupby(['planning_year']).agg(
    annual_total_mw=('cleared_volume', 'sum'),
    annual_total_mwh=('cleared_mwh', 'sum'),
    annual_rows=('path', 'size'),
    annual_unique_paths=('path', 'nunique'),
    annual_avg_mw_per_row=('cleared_volume', 'mean'),
).reset_index()

f0p_vol = f0p_all.groupby(['planning_year']).agg(
    f0p_total_mw=('cleared_volume', 'sum'),
    f0p_total_mwh=('cleared_mwh', 'sum'),
    f0p_rows=('path', 'size'),
    f0p_unique_paths=('path', 'nunique'),
    f0p_avg_mw_per_row=('cleared_volume', 'mean'),
).reset_index()

comparison = annual_vol.merge(f0p_vol, on='planning_year', how='outer')
comparison['mw_ratio_annual_over_f0p'] = comparison['annual_avg_mw_per_row'] / comparison['f0p_avg_mw_per_row']
comparison['total_mw_ratio'] = comparison['annual_total_mw'] / comparison['f0p_total_mw']

print('=== Annual vs f0p Volume Comparison ===')
print(comparison.to_string(index=False))
print()
print('--- Averages across all PYs ---')
print(comparison[['annual_avg_mw_per_row','f0p_avg_mw_per_row','mw_ratio_annual_over_f0p','total_mw_ratio']].mean())

In [ ]:
# More granular: compare per-path average MW by period_type
annual_per_path = annual_all.groupby(['period_type', 'class_type'])['cleared_volume'].agg(['mean','median','sum','count']).reset_index()
annual_per_path.columns = ['period_type','class_type','annual_mean_mw','annual_median_mw','annual_total_mw','annual_count']

f0p_per_path = f0p_all.groupby(['period_type', 'class_type'])['cleared_volume'].agg(['mean','median','sum','count']).reset_index()
f0p_per_path.columns = ['period_type','class_type','f0p_mean_mw','f0p_median_mw','f0p_total_mw','f0p_count']

print('=== Annual per-path MW by period_type x class_type ===')
print(annual_per_path.to_string(index=False))
print()
print('=== f0p per-path MW by period_type x class_type ===')
print(f0p_per_path.to_string(index=False))

In [ ]:
# MCP distribution comparison: annual vs f0p
print('=== Annual MCP Distribution ===')
print(annual_all.groupby(['round','period_type'])['mcp'].describe().to_string())
print()
print('=== f0p MCP Distribution ===')
print(f0p_all.groupby(['period_type'])['mcp'].describe().to_string())

In [ ]:
# Save annual_all and f0p_all for notebook 02
os.makedirs('/opt/temp/qianli/annual_research/', exist_ok=True)
annual_all.to_parquet('/opt/temp/qianli/annual_research/annual_cleared_all.parquet', index=False)
f0p_all.to_parquet('/opt/temp/qianli/annual_research/f0p_cleared_all.parquet', index=False)
print('Saved annual_all and f0p_all to /opt/temp/qianli/annual_research/')